# Lab 2: Pandas for Cat and Dog Faces

This notebook builds an image metadata table from the dataset folders and uses Pandas to inspect, audit, summarize, and sample the data.

The implementations below follow the required function names so the notebook can be converted to `notebook.py` and checked by the provided tests.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 1234
GENERATED_METADATA_PATH = ARTIFACT_DIR / "lab2_faces_metadata.csv"

SPLITS = ("train", "val", "test")
LABELS = ("cat", "dog")
IMAGE_EXTENSIONS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")
ALLOWED_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"Dataset root : {DATA_ROOT}")
print(f"Metadata path: {GENERATED_METADATA_PATH}")

## Question 1: Build metadata from folders

For every split and label, find supported image files, inspect each image, and create one metadata row. File paths are stored relative to `DATA_ROOT`.

In [ ]:
def list_image_paths_for_group(data_root: Path, split: str, label: str) -> list[Path]:
    group_directory = Path(data_root) / split / label
    if not group_directory.is_dir():
        return []

    image_paths = [
        candidate
        for candidate in group_directory.iterdir()
        if candidate.is_file() and candidate.suffix.lower() in ALLOWED_SUFFIXES
    ]
    return sorted(image_paths)


def inspect_image_file(path: Path) -> tuple[int, int, float]:
    with Image.open(path) as source_image:
        rgb_image = source_image.convert("RGB")
        width, height = rgb_image.size
        normalized_pixels = np.asarray(rgb_image, dtype=np.float32) / 255.0

    return width, height, float(normalized_pixels.mean())


def make_metadata_row(
    path: Path, data_root: Path, split: str, label: str
) -> dict[str, object]:
    width, height, mean_intensity = inspect_image_file(path)
    relative_path = path.relative_to(data_root)

    return {
        "filepath": str(relative_path),
        "label": label,
        "split": split,
        "width": width,
        "height": height,
        "mean_intensity": mean_intensity,
    }


def build_metadata_from_folders(data_root: Path) -> pd.DataFrame:
    records: list[dict[str, object]] = []

    for split in SPLITS:
        for label in LABELS:
            group_paths = list_image_paths_for_group(data_root, split, label)
            for image_path in group_paths:
                records.append(
                    make_metadata_row(image_path, data_root, split, label)
                )

    columns = [
        "filepath",
        "label",
        "split",
        "width",
        "height",
        "mean_intensity",
    ]
    metadata = pd.DataFrame.from_records(records, columns=columns)

    if metadata.empty:
        return metadata

    return metadata.sort_values(
        ["split", "label", "filepath"]
    ).reset_index(drop=True)


folder_df = build_metadata_from_folders(DATA_ROOT)
print("metadata shape:", folder_df.shape)
display(folder_df.head())

folder_df.to_csv(GENERATED_METADATA_PATH, index=False)
print(f"Saved metadata to: {GENERATED_METADATA_PATH}")

## Question 2: Load the saved metadata

Read the generated CSV into a Pandas DataFrame.

In [ ]:
def load_metadata_table(csv_path: Path) -> pd.DataFrame:
    return pd.read_csv(csv_path)


df = load_metadata_table(GENERATED_METADATA_PATH)
print("loaded shape:", df.shape)
display(df.head())

## Question 3: Inspect the metadata table

Return the row count, column names, class counts, and split counts.

In [ ]:
def summarize_metadata(frame: pd.DataFrame) -> dict[str, object]:
    return {
        "rows": len(frame),
        "columns": frame.columns.to_list(),
        "class_counts": frame["label"].value_counts(),
        "split_counts": frame["split"].value_counts(),
    }


summary = summarize_metadata(df)
print("Rows:", summary["rows"])
print("Columns:", summary["columns"])
print("\nClass counts:\n", summary["class_counts"])
print("\nSplit counts:\n", summary["split_counts"])


## Question 4: Count samples per split and class

Create a table with labels as rows, splits as columns, and sample counts as values.

In [ ]:
def build_label_split_table(frame: pd.DataFrame) -> pd.DataFrame:
    return pd.crosstab(frame["label"], frame["split"])


label_split_table = build_label_split_table(df)
display(label_split_table)

## Question 5: Audit metadata quality

Check missing values, duplicate file paths, unexpected labels, and invalid image dimensions.

In [ ]:
def audit_metadata(frame: pd.DataFrame) -> dict[str, object]:
    missing_by_column = frame.isna().sum().astype(int).to_dict()
    duplicate_count = int(frame["filepath"].duplicated().sum())

    invalid_label_mask = ~frame["label"].isin(LABELS)
    invalid_labels = sorted(frame.loc[invalid_label_mask, "label"].dropna().unique().tolist())

    invalid_size_mask = (frame["width"] <= 0) | (frame["height"] <= 0)
    invalid_size_count = int(invalid_size_mask.sum())

    return {
        "missing_values": missing_by_column,
        "duplicate_filepaths": duplicate_count,
        "bad_labels": invalid_labels,
        "non_positive_sizes": invalid_size_count,
    }


audit_report = audit_metadata(df)
print(audit_report)

## Question 6: Add analysis columns

Add pixel count, aspect ratio, brightness quartile, and a size category relative to a 64 by 64 image.

In [ ]:
def add_analysis_columns(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    result["pixel_count"] = result["width"] * result["height"]
    result["aspect_ratio"] = result["width"] / result["height"]

    brightness_labels = ["darkest", "dim", "bright", "brightest"]
    result["brightness_band"] = pd.qcut(
        result["mean_intensity"],
        q=4,
        labels=brightness_labels,
    )

    reference_pixels = 64 * 64
    result["size_bucket"] = np.select(
        [
            result["pixel_count"] < reference_pixels,
            result["pixel_count"] > reference_pixels,
        ],
        ["small", "large"],
        default="medium",
    )

    return result


analysis_df = add_analysis_columns(df)
display(analysis_df.head())

## Question 7: Compare image characteristics across splits

Group the analysis table by split and calculate the requested averages.

In [ ]:
def build_split_characteristics_table(frame: pd.DataFrame) -> pd.DataFrame:
    source_columns = ["width", "height", "pixel_count", "mean_intensity"]

    return (
        frame.groupby("split")[source_columns]
        .mean()
        .rename(
            columns={
                "width": "avg_width",
                "height": "avg_height",
                "pixel_count": "avg_pixel_count",
                "mean_intensity": "avg_mean_intensity",
            }
        )
    )


split_characteristics = build_split_characteristics_table(analysis_df)
display(split_characteristics)

## Question 8: Create a balanced sample

Take up to `n_per_group` rows independently from every `(split, label)` group.

In [ ]:
def sample_balanced_by_split_and_label(
    frame: pd.DataFrame, n_per_group: int, seed: int
) -> pd.DataFrame:
    if n_per_group < 0:
        raise ValueError("n_per_group must be non-negative")

    sampled_groups: list[pd.DataFrame] = []

    for _, group in frame.groupby(["split", "label"], sort=True):
        rows_to_take = min(n_per_group, len(group))
        sampled_groups.append(
            group.sample(n=rows_to_take, random_state=seed)
        )

    if not sampled_groups:
        return frame.iloc[0:0].copy().reset_index(drop=True)

    return pd.concat(sampled_groups, ignore_index=True)


sample_size_per_group = 5
sampled_df = sample_balanced_by_split_and_label(
    analysis_df,
    n_per_group=sample_size_per_group,
    seed=SEED,
)
print("sampled shape:", sampled_df.shape)
display(sampled_df.head())
display(pd.crosstab(sampled_df["label"], sampled_df["split"]))

## Reflection

1. Common alternatives include 70/15/15 and 80/10/10 train-validation-test splits.
2. For a very small dataset, cross-validation can use the available observations more efficiently than one fixed validation split.
3. Grouped sampling keeps important subgroups represented in a smaller experimental dataset.
4. `groupby` is useful because it supports both grouped summaries and group-wise sampling.